# kvpress runs: SnapKV, StreamingLLM, Random, H2O (ObservedAttention)

Runs `scripts/run_kvpress.py` for every method at budgets 0.2, 0.1 and 0.05 on the 169 questions. The choices behind the runner are in `docs/runner_notes.md`, the output files in `docs/kvpress_fields.md`.

Order: checks first (cells 7–10), then the full runs (cell 11). Every finished run is copied to Google Drive straight away. After a disconnect, run all cells again: runs already on Drive are skipped.

In [ ]:
# 1. Check the GPU: all team runs use the same GPU type (L4)
import torch
gpu = torch.cuda.get_device_properties(0)
print(gpu.name, f"{gpu.total_memory / 2**30:.1f} GiB")
assert "L4" in gpu.name, "Switch the runtime to an L4 GPU (Runtime > Change runtime type)"

In [ ]:
# 2. Settings
BRANCH = "kv-press"   # the branch with scripts/run_kvpress.py; "main" once it is merged
METHODS = ["snapkv", "streamingllm", "random", "h2o"]
BUDGETS = [0.2, 0.1, 0.05]
DRIVE_DIR = "/content/drive/MyDrive/kv-evidence-loss"   # masks in raw/kept/ (team layout), answers in answers/

In [ ]:
# 3. Get a fresh copy of the branch
%cd /content
!rm -rf kv-evidence-loss
!git clone -q -b {BRANCH} https://github.com/burakzor/kv-evidence-loss.git
%cd /content/kv-evidence-loss
!git log -1 --format="%h %ad %s" --date=short

In [ ]:
# 4. Install the pinned team environment (torch is Colab's own CUDA build)
!pip install -q -r requirements.txt
!pip list 2>/dev/null | grep -iE "^(torch|transformers|tokenizers|kvpress|accelerate|datasets|pandas|numpy|pyarrow|tqdm) "

In [ ]:
# 5. Optional: Hugging Face token for a faster download (only works in the Colab web UI)
import os
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print(f"No HF token ({type(e).__name__}); downloading anonymously.")

In [ ]:
# 6. Drive, environment record and helpers
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path(DRIVE_DIR)
ANSWERS = Path("results/answers")

# The environment is the same for every run of this session; each run gets its own copy, {method}_{budget}_env.txt
!(nvidia-smi --query-gpu=name,driver_version --format=csv,noheader; python --version; python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda)"; pip freeze) > /content/env.txt
!head -4 /content/env.txt

def run_name(method, budget, limit=None):
    return f"{method}_{float(budget)}" + (f"_limit{limit}" if limit else "")

def run(method, budget, limit=None, extra=""):
    """Run the runner once; stop the notebook if it fails. Returns the answers table."""
    name = run_name(method, budget, limit)
    out = ANSWERS / f"{name}.parquet"
    out.unlink(missing_ok=True)
    cmd = f"python scripts/run_kvpress.py --method {method} --budget {budget}" + (f" --limit {limit}" if limit else "") + (f" {extra}" if extra else "")
    print("$", cmd)
    get_ipython().system(cmd)
    if not out.exists():
        raise RuntimeError(f"{name} failed: see the error above")
    shutil.copy("/content/env.txt", ANSWERS / f"{name}_env.txt")
    return pd.read_parquet(out)

def save_to_drive(method, budget):
    """Copy one finished run (answers, config, env, masks) to Drive."""
    name = run_name(method, budget)
    (DRIVE / "answers").mkdir(parents=True, exist_ok=True)
    for suffix in [".parquet", "_config.json", "_env.txt"]:
        shutil.copy(ANSWERS / f"{name}{suffix}", DRIVE / "answers")
    masks = Path("results/raw/kept") / method / str(float(budget))
    if masks.exists():  # nopress writes no masks
        shutil.copytree(masks, DRIVE / "raw" / "kept" / method / str(float(budget)), dirs_exist_ok=True)
    print("saved to Drive:", name)

def done_on_drive(method, budget):
    """True if this run is complete on Drive: answers for 169 questions and, except for nopress, 169 masks."""
    answers = DRIVE / "answers" / f"{run_name(method, budget)}.parquet"
    if not answers.exists() or len(pd.read_parquet(answers)) != 169:
        return False
    return method == "nopress" or len(list((DRIVE / "raw" / "kept" / method / str(float(budget))).glob("*.npz"))) == 169

In [ ]:
# 7. Test: one question end to end (first run downloads Qwen3-8B, ~16 GB); writes *_limit1 files and results/raw/kept_limit1/
# Look at the printout: kept tokens of layer 15, head 0, and how much of the A and B sentences survived
df = run("snapkv", 0.1, limit=1, extra="--inspect-layer 15")
df.T

In [ ]:
# 8. Budget 1.0 check: nothing is removed, so the answers must equal the full-cache baseline
if done_on_drive("nopress", 1.0):
    nopress = pd.read_parquet(DRIVE / "answers" / "nopress_1.0.parquet")
else:
    nopress = run("nopress", 1.0)
    save_to_drive("nopress", 1.0)
full = pd.read_parquet(ANSWERS / "full_1.0.parquet")
assert (nopress["kept_fraction"] == 1.0).all()
same = nopress["answer"].eq(full.loc[nopress.index, "answer"])
print(f"answers equal to full_1.0: {same.sum()}/{len(same)}")
print("contains_alias:", round(full["contains_alias"].mean(), 3), "(baseline) vs", round(nopress["contains_alias"].mean(), 3), "(nopress)")
pd.concat([full.loc[~same, "answer"], nopress.loc[~same, "answer"]], axis=1, keys=["full", "nopress"])

In [ ]:
# 9. Random check: the mean kept fraction must equal the budget (up to rounding to whole tokens)
df = run("random", 0.1, limit=10)
print("mean kept_fraction:", round(df["kept_fraction"].mean(), 4))
assert abs(df["kept_fraction"].mean() - 0.1) < 0.002

In [ ]:
# 10. H2O check: eager attention runs and fits in memory (the full run then covers the longest prompt, 2,683 tokens)
df = run("h2o", 0.1, limit=1)
print("peak memory:", round(df["peak_mem_gb"].max(), 2), "GB of", round(gpu.total_memory / 2**30, 1))

In [ ]:
# 11. Full runs: 4 methods x 3 budgets, 169 questions each; each run is saved to Drive when it finishes
for method in METHODS:
    for budget in BUDGETS:
        if done_on_drive(method, budget):
            print("already on Drive, skipping:", run_name(method, budget))
            continue
        run(method, budget)
        save_to_drive(method, budget)

In [ ]:
# 12. Results: accuracy (all questions and the model-filter keep set), kept share, memory and time
keep = pd.read_parquet("data/hotpotqa/filter_qwen3_8b.parquet")["keep"]
def summary(df):
    return {"contains_alias": df["contains_alias"].mean(), "contains_alias_keep": df.loc[keep[keep].index.intersection(df.index), "contains_alias"].mean(),
            "f1": df["f1"].mean(), "kept_fraction": df.get("kept_fraction", pd.Series([1.0])).mean(), "kv_cache_mb": df.get("kv_cache_mb", pd.Series([np.nan])).mean(),
            "prefill_s": df.get("prefill_s", pd.Series([np.nan])).mean(), "decode_ms_per_token": 1000 * df.get("decode_s_per_token", pd.Series([np.nan])).mean(),
            "peak_mem_gb": df.get("peak_mem_gb", pd.Series([np.nan])).max()}
rows = {"full_1.0 (baseline)": summary(full)}
for name in [run_name("nopress", 1.0)] + [run_name(m, b) for m in METHODS for b in BUDGETS]:
    path = DRIVE / "answers" / f"{name}.parquet"
    if path.exists():
        rows[name] = summary(pd.read_parquet(path))
pd.DataFrame(rows).T.round(3)

In [ ]:
# 13. Quick look at RQ1: share of the A and B sentences kept, per method and budget (the team's numbers come from scripts/retention.py)
import sys
sys.path.insert(0, "scripts")
from retention import get_retention
spans = pd.read_parquet("data/hotpotqa/hotpot_token_spans_qwen3_8b.parquet")
parts = []
for method in METHODS:
    for budget in BUDGETS:
        for path in sorted((DRIVE / "raw" / "kept" / method / str(float(budget))).glob("*.npz")):
            parts.append(get_retention(np.load(path)["kept"], spans.loc[[path.stem]]).assign(method=method, budget=budget))
pd.concat(parts).groupby(["method", "budget", "sentence_tag"])["kept_fraction"].mean().unstack().round(3)

In [ ]:
# 14. Download the answers, configs and environment records in one zip (the masks stay on Drive: results/raw/ is not in git)
# Unzip into results/answers/ of your local repo and commit them
import zipfile
from google.colab import files
names = [run_name("nopress", 1.0)] + [run_name(m, b) for m in METHODS for b in BUDGETS]
with zipfile.ZipFile("/content/kvpress_answers.zip", "w") as z:
    for name in names:
        for suffix in [".parquet", "_config.json", "_env.txt"]:
            path = DRIVE / "answers" / f"{name}{suffix}"
            if path.exists():
                z.write(path, arcname=path.name)
print(len(zipfile.ZipFile("/content/kvpress_answers.zip").namelist()), "files")
files.download("/content/kvpress_answers.zip")